# 03 Modelo Clasificacion Ticket Alto

Este notebook usa la base `EDA`, entrena una `Regresion Logistica`, genera predicciones y exporta el `Parquet` final de clasificacion.


## 1. Librerias


In [ ]:
from pathlib import Path
import json

import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## 2. Definir rutas y cargar la base EDA


In [ ]:
from pathlib import Path

def resolve_project_root() -> Path:
    # Buscar la raiz del proyecto a partir del directorio actual.
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "parquets").exists():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto")

PROJECT_ROOT = resolve_project_root()
PROJECT_ROOT

# Definir las rutas de entrada y salida de la etapa 03.
INPUT_PATH = PROJECT_ROOT / "parquets" / "02_EDA_Base_Tickets" / "02_base_eda_tickets.parquet"
OUTPUT_DIR = PROJECT_ROOT / "parquets" / "03_Modelo_Clasificacion_Ticket_Alto"
OUTPUT_PATH = OUTPUT_DIR / "03_tickets_clasificados.parquet"
MODEL_DIR = PROJECT_ROOT / "models" / "03_Modelo_Clasificacion_Ticket_Alto"
MODEL_PATH = MODEL_DIR / "03_modelo_regresion_logistica.joblib"
FEATURES_PATH = MODEL_DIR / "03_features_regresion_logistica.json"

# Leer la base EDA de entrada.
df = pd.read_parquet(INPUT_PATH)
df.head()


### Resultado breve

La etapa `03` parte de la base exploratoria ya validada y usa esa estructura para aprender si un ticket puede clasificarse como alto.


## 3. Seleccionar variables y target


In [ ]:
# Definir las variables explicativas del modelo.
FEATURES = [
    "dia", "mes", "trimestre", "dia_semana", "dia_tipo", "fin_semana",
    "ciudad", "capacidad_sucursal", "tipo_empleado", "salario", "turno",
    "numero_mesa", "capacidad_mesa", "metodo_pago", "lineas_ticket",
    "cantidad_total", "platillos_distintos", "categorias_distintas",
    "incluye_bebida", "incluye_postre", "incluye_entrada", "incluye_platillo_fuerte"
]

# Definir la variable objetivo.
TARGET = "ticket_alto"

# Separar X y y para el entrenamiento.
X = df[FEATURES].copy()
y = df[TARGET].copy()
X.head()


## 4. Particion de entrenamiento y prueba


In [ ]:
# Dividir la base en entrenamiento y prueba con estratificacion.
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42, stratify=y
)

# Revisar dimensiones de las particiones.
X_train.shape, X_test.shape


## 5. Entrenar y evaluar el modelo


In [ ]:
# Separar columnas numericas y categoricas.
numericas = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categoricas = X.select_dtypes(include=["object"]).columns.tolist()

# Definir el preprocesador del pipeline.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numericas,
        ),
        (
            "cat",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categoricas,
        ),
    ]
)

# Definir el pipeline completo de clasificacion.
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

# Entrenar el modelo con la muestra de entrenamiento.
pipeline.fit(X_train, y_train)

# Generar predicciones y probabilidades sobre test.
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

# Resumir metricas principales.
metricas = pd.DataFrame(
    {
        "metrica": ["accuracy", "precision", "recall", "f1", "roc_auc"],
        "valor": [
            round(float(accuracy_score(y_test, y_pred)), 4),
            round(float(precision_score(y_test, y_pred)), 4),
            round(float(recall_score(y_test, y_pred)), 4),
            round(float(f1_score(y_test, y_pred)), 4),
            round(float(roc_auc_score(y_test, y_prob)), 4),
        ],
    }
)
metricas


### Resultado breve

Las metricas permiten revisar que tan bien separa el modelo los tickets altos frente a los normales antes de exportar la salida final.


## 6. Visualizaciones del modelo


In [ ]:
# Construir una vista rapida de resultados sobre test.
resultados_test = X_test.copy()
resultados_test["ticket_alto_real"] = y_test
resultados_test["ticket_alto_pred"] = y_pred
resultados_test["prob_ticket_alto"] = y_prob

# Graficar la matriz de confusion y la distribucion de probabilidades.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Matriz de confusion")

sns.histplot(
    data=resultados_test,
    x="prob_ticket_alto",
    hue="ticket_alto_real",
    bins=20,
    kde=True,
    stat="density",
    common_norm=False,
    ax=axes[1],
)
axes[1].set_title("Probabilidad estimada de ticket alto")
axes[1].set_xlabel("prob_ticket_alto")
plt.tight_layout()


In [ ]:
# Graficar la curva ROC del modelo.
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
ax.set_title("Curva ROC del modelo de clasificacion")
plt.tight_layout()


### Resultado breve

Las visualizaciones permiten ver la separacion del modelo, los aciertos y errores principales y la forma en que distribuye la probabilidad de ticket alto.


## 7. Exportar el parquet final del modelo


In [ ]:
# Crear carpetas de salida para la etapa 03.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Entrenar el modelo final con toda la base disponible.
final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced")),
])
final_pipeline.fit(X, y)

# Generar la salida final con predicciones y probabilidades por ticket.
df_predictions = df.copy()
df_predictions["pred_ticket_alto"] = final_pipeline.predict(X)
df_predictions["prob_ticket_alto"] = final_pipeline.predict_proba(X)[:, 1]
df_predictions["modelo"] = "Regresion Logistica"
df_predictions["fuente_parquet"] = "02_base_eda_tickets.parquet"
df_predictions["conjunto_evaluacion"] = "train"
df_predictions.loc[idx_test, "conjunto_evaluacion"] = "test"

test_lookup = pd.DataFrame(
    {
        "idx": idx_test,
        "pred_ticket_alto_test": y_pred,
        "prob_ticket_alto_test": y_prob,
    }
).set_index("idx")

df_predictions = df_predictions.join(test_lookup, how="left")

# Exportar el parquet final y los artefactos del modelo.
df_predictions.to_parquet(OUTPUT_PATH, index=False)
joblib.dump(final_pipeline, MODEL_PATH)
FEATURES_PATH.write_text(json.dumps(FEATURES, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Parquet generado en: {OUTPUT_PATH}")
print(f"Modelo guardado en: {MODEL_PATH}")


In [ ]:
# Verificar la salida exportada de la etapa 03.
df_exportado = pd.read_parquet(OUTPUT_PATH)
df_exportado.shape


### Resultado breve

La etapa `03` queda cerrada cuando el parquet de clasificacion ya incluye la clase predicha, la probabilidad y la marca de evaluacion para cada ticket.
